# 🧠 Positive Patterns Analysis: Cognitive & Sentiment Probes

This notebook analyzes the positive_patterns.jsonl dataset using both cognitive and sentiment probes.

**Analysis includes:**
- Comparison of positive vs negative vs transformation patterns
- Sentiment analysis across pattern types
- Cognitive action patterns and their combinations with sentiment
- Streaming vs whole string inference comparison
- Interactive HTML visualization

**Requirements:**
- GPU with CUDA support (or ROCm for AMD)
- ~8GB VRAM minimum
- PyTorch, transformers, h5py, numpy

## 1. Setup Environment

In [ ]:
# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Check GPU availability
import torch
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ Warning: No GPU detected. This will be very slow on CPU.")

In [ ]:
%%bash
# Install dependencies if in Colab
if [ "$IN_COLAB" = "True" ]; then
    echo "Installing dependencies..."
    pip install -q torch transformers h5py numpy scikit-learn tqdm
    echo "✓ Dependencies installed"
fi

## 2. Clone Repository and Setup

In [ ]:
import os
from pathlib import Path

# Setup paths
if IN_COLAB:
    REPO_DIR = Path("/content/brije")
    
    # Clone repo if not exists
    if not REPO_DIR.exists():
        print("Cloning repository...")
        !git clone https://github.com/ChuloIva/brije.git /content/brije
        print("✓ Repository cloned")
    else:
        print("✓ Repository already exists")
    
    # Change to repo directory
    os.chdir(REPO_DIR)
else:
    # Assume running from notebooks directory
    REPO_DIR = Path.cwd().parent
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")

# Add source to path
import sys
sys.path.insert(0, str(REPO_DIR / "src" / "probes"))
sys.path.insert(0, str(REPO_DIR / "third_party" / "nnsight" / "src"))

print("✓ Environment setup complete")

## 3. Check Data Files

Verify that all required data files are present:
- Dataset: `data/positive_patterns.jsonl`
- Cognitive probes: `data/probes_binary/layer_*/probe_*.pth`
- Sentiment probes: `data/sentiment/layer_*/sentiment_regression_probe.pth`

In [ ]:
# Check data files
data_path = REPO_DIR / "data" / "positive_patterns.jsonl"
probes_dir = REPO_DIR / "data" / "probes_binary"
sentiment_dir = REPO_DIR / "data" / "sentiment"

print("Checking data files...\n")

# Check dataset
if data_path.exists():
    import json
    with open(data_path) as f:
        num_entries = sum(1 for _ in f)
    print(f"✓ Dataset: {num_entries} entries")
else:
    print(f"✗ Dataset not found at {data_path}")

# Check cognitive probes
if probes_dir.exists():
    probe_files = list(probes_dir.glob("layer_*/probe_*.pth"))
    layers = set([p.parent.name for p in probe_files])
    print(f"✓ Cognitive probes: {len(probe_files)} probes across {len(layers)} layers")
    print(f"  Layers: {sorted(layers, key=lambda x: int(x.split('_')[1]))}")
else:
    print(f"✗ Cognitive probes not found at {probes_dir}")

# Check sentiment probes
if sentiment_dir.exists():
    sent_files = list(sentiment_dir.glob("layer_*/sentiment_regression_probe.pth"))
    sent_layers = set([p.parent.name for p in sent_files])
    print(f"✓ Sentiment probes: {len(sent_files)} probes across {len(sent_layers)} layers")
    print(f"  Layers: {sorted(sent_layers, key=lambda x: int(x.split('_')[1]))}")
else:
    print(f"✗ Sentiment probes not found at {sentiment_dir}")

print("\n✓ All data files verified")

## 4. Import Analysis Modules

In [ ]:
# Import GPU configuration
try:
    from gpu_utils import configure_amd_gpu
    configure_amd_gpu()
except:
    print("Note: AMD GPU configuration not available, using default CUDA")

# Import analysis modules
from streaming_probe_inference import StreamingProbeInferenceEngine, AggregatedPrediction
from collections import Counter, defaultdict
import numpy as np
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

print("✓ Analysis modules imported")

## 5. Configuration

In [ ]:
# Analysis configuration
CONFIG = {
    'data_path': data_path,
    'probes_base_dir': probes_dir,
    'sentiment_probes_dir': sentiment_dir,
    'model_name': 'google/gemma-3-4b-it',
    'layer_range': (15, 30),  # Cognitive probe layers
    'threshold': 0.5,  # Activation threshold
    'output_dir': REPO_DIR / 'results' / 'positive_patterns_analysis'
}

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 6. Load Dataset

In [ ]:
def load_dataset(file_path: Path) -> List[Dict]:
    """Load positive_patterns.jsonl"""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

print("Loading dataset...")
dataset = load_dataset(CONFIG['data_path'])
print(f"✓ Loaded {len(dataset)} entries")

# Show sample entry
print("\nSample entry keys:")
print(list(dataset[0].keys()))

## 7. Initialize Inference Engine

This will load the language model and all probes. **This step takes 2-3 minutes.**

In [ ]:
print("Initializing inference engine...")
print("This will load the model and all probes (2-3 minutes)\n")

engine = StreamingProbeInferenceEngine(
    probes_base_dir=CONFIG['probes_base_dir'],
    model_name=CONFIG['model_name'],
    sentiment_probes_dir=CONFIG['sentiment_probes_dir'],
    include_sentiment=True,
    layer_range=CONFIG['layer_range'],
    verbose=True
)

print(f"\n✓ Engine initialized")
print(f"  Cognitive probes: {len(engine.probes)}")
print(f"  Sentiment probes: {len(engine.sentiment_probes)}")
print(f"  Layers: {engine.layer_start}-{engine.layer_end}")

## 8. Define Analysis Functions

In [ ]:
@dataclass
class PatternAnalysis:
    """Analysis results for a single pattern"""
    pattern_type: str
    text: str
    cognitive_pattern_name: str
    cognitive_pattern_type: str
    streaming_top_actions: List[Tuple[str, int, float]]
    streaming_sentiment_avg: float
    streaming_sentiment_layers: List[int]
    whole_top_actions: List[Tuple[str, int, float]]
    whole_sentiment_avg: float
    whole_sentiment_layers: List[int]
    action_agreement: float
    sentiment_diff: float


def extract_pattern_texts(entry: Dict) -> Dict[str, str]:
    """Extract positive, negative, and transformation pattern texts"""
    return {
        'positive': entry.get('positive_thought_pattern', ''),
        'negative': entry.get('reference_negative_example', ''),
        'transformation': entry.get('reference_transformed_example', '')
    }


def run_inference(engine, text: str, threshold: float):
    """Run inference and return cognitive + sentiment predictions"""
    total_probes = len(engine.probes) + len(engine.sentiment_probes)
    
    all_predictions = engine.predict_streaming(
        text,
        top_k=total_probes,
        threshold=0.0,
        show_realtime=False
    )
    
    aggregated = engine.aggregate_predictions(all_predictions, threshold=threshold)
    
    cognitive_preds = [p for p in aggregated if p.action_name != "sentiment"]
    sentiment_preds = [p for p in aggregated if p.action_name == "sentiment"]
    
    cognitive_preds.sort(key=lambda x: (x.layer_count, x.max_confidence), reverse=True)
    
    return cognitive_preds, sentiment_preds


def analyze_pattern(engine, pattern_type: str, text: str, entry: Dict, threshold: float) -> PatternAnalysis:
    """Analyze single pattern"""
    streaming_cog, streaming_sent = run_inference(engine, text, threshold)
    whole_cog, whole_sent = run_inference(engine, text, threshold)
    
    streaming_top = [(p.action_name, p.layer_count, p.max_confidence) 
                     for p in streaming_cog[:10] if p.is_active]
    whole_top = [(p.action_name, p.layer_count, p.max_confidence) 
                 for p in whole_cog[:10] if p.is_active]
    
    streaming_sent_avg = np.mean([p.max_confidence for p in streaming_sent]) if streaming_sent else 0.0
    streaming_sent_layers = [p.best_layer for p in streaming_sent if p.is_active]
    
    whole_sent_avg = np.mean([p.max_confidence for p in whole_sent]) if whole_sent else 0.0
    whole_sent_layers = [p.best_layer for p in whole_sent if p.is_active]
    
    streaming_actions = set([a[0] for a in streaming_top[:5]])
    whole_actions = set([a[0] for a in whole_top[:5]])
    
    if len(streaming_actions) == 0 and len(whole_actions) == 0:
        action_agreement = 1.0
    else:
        action_agreement = len(streaming_actions & whole_actions) / len(streaming_actions | whole_actions) if (streaming_actions | whole_actions) else 0.0
    
    sentiment_diff = abs(streaming_sent_avg - whole_sent_avg)
    
    return PatternAnalysis(
        pattern_type=pattern_type,
        text=text[:200],
        cognitive_pattern_name=entry.get('cognitive_pattern_name', ''),
        cognitive_pattern_type=entry.get('cognitive_pattern_type', ''),
        streaming_top_actions=streaming_top,
        streaming_sentiment_avg=streaming_sent_avg,
        streaming_sentiment_layers=streaming_sent_layers,
        whole_top_actions=whole_top,
        whole_sentiment_avg=whole_sent_avg,
        whole_sentiment_layers=whole_sent_layers,
        action_agreement=action_agreement,
        sentiment_diff=sentiment_diff
    )

print("✓ Analysis functions defined")

## 9. Run Analysis

**This is the main analysis step.** It processes all patterns in the dataset.

**Time estimate:**
- 520 entries × 3 patterns = 1560 total analyses
- ~2-5 seconds per pattern on GPU
- **Total time: 1-2 hours**

For testing, set `MAX_ENTRIES` to a small number (e.g., 10)

In [ ]:
# Set to None to process all, or a number for testing (e.g., 10)
MAX_ENTRIES = None  # Change to 10 for quick test

print("Starting analysis...")
if MAX_ENTRIES:
    print(f"TEST MODE: Processing only {MAX_ENTRIES} entries")
    dataset_subset = dataset[:MAX_ENTRIES]
else:
    print(f"FULL MODE: Processing all {len(dataset)} entries")
    dataset_subset = dataset

print(f"\nExpected analyses: ~{len(dataset_subset) * 3}")
print("\nProgress:\n")

all_analyses = []

for idx, entry in enumerate(dataset_subset):
    if idx % 10 == 0:
        print(f"  [{idx}/{len(dataset_subset)}] ({idx/len(dataset_subset)*100:.1f}%) - Completed: {len(all_analyses)} analyses")
    
    patterns = extract_pattern_texts(entry)
    
    for pattern_type, text in patterns.items():
        if not text.strip():
            continue
        
        try:
            analysis = analyze_pattern(
                engine,
                pattern_type,
                text,
                entry,
                CONFIG['threshold']
            )
            all_analyses.append(analysis)
        except Exception as e:
            print(f"  ⚠️ Failed to analyze {pattern_type} pattern #{idx}: {e}")
            continue

print(f"\n✓ Analysis complete! Processed {len(all_analyses)} patterns")

## 10. Compute Group Statistics

In [ ]:
def aggregate_group_statistics(analyses: List[PatternAnalysis], pattern_type: str) -> Dict:
    """Compute statistics for a pattern type group"""
    group_analyses = [a for a in analyses if a.pattern_type == pattern_type]
    
    if not group_analyses:
        return {}
    
    streaming_action_counts = Counter()
    for analysis in group_analyses:
        for action, layer_count, conf in analysis.streaming_top_actions:
            streaming_action_counts[action] += 1
    
    whole_action_counts = Counter()
    for analysis in group_analyses:
        for action, layer_count, conf in analysis.whole_top_actions:
            whole_action_counts[action] += 1
    
    streaming_sentiments = [a.streaming_sentiment_avg for a in group_analyses]
    whole_sentiments = [a.whole_sentiment_avg for a in group_analyses]
    agreements = [a.action_agreement for a in group_analyses]
    sentiment_diffs = [a.sentiment_diff for a in group_analyses]
    
    return {
        'pattern_type': pattern_type,
        'num_samples': len(group_analyses),
        'top_streaming_actions': streaming_action_counts.most_common(15),
        'top_whole_actions': whole_action_counts.most_common(15),
        'avg_sentiment_streaming': float(np.mean(streaming_sentiments)),
        'std_sentiment_streaming': float(np.std(streaming_sentiments)),
        'avg_sentiment_whole': float(np.mean(whole_sentiments)),
        'std_sentiment_whole': float(np.std(whole_sentiments)),
        'avg_action_agreement': float(np.mean(agreements)),
        'avg_sentiment_diff': float(np.mean(sentiment_diffs)),
    }

print("Computing group statistics...")
pattern_types = ['positive', 'negative', 'transformation']
group_stats = {}

for pt in pattern_types:
    stats = aggregate_group_statistics(all_analyses, pt)
    if stats:
        group_stats[pt] = stats
        print(f"  {pt.upper()}: {stats['num_samples']} samples")

print("\n✓ Group statistics computed")

## 11. Display Results Summary

In [ ]:
print("=" * 80)
print("ANALYSIS RESULTS SUMMARY")
print("=" * 80)

for pt, stats in group_stats.items():
    print(f"\n{pt.upper()} Patterns ({stats['num_samples']} samples):")
    print(f"  Sentiment (streaming): {stats['avg_sentiment_streaming']:.3f} ± {stats['std_sentiment_streaming']:.3f}")
    print(f"  Sentiment (whole): {stats['avg_sentiment_whole']:.3f} ± {stats['std_sentiment_whole']:.3f}")
    print(f"  Action agreement: {stats['avg_action_agreement']:.1%}")
    print(f"\n  Top 5 cognitive actions:")
    for i, (action, count) in enumerate(stats['top_streaming_actions'][:5], 1):
        print(f"    {i}. {action}: {count}")

print("\n" + "=" * 80)

## 12. Visualization: Sentiment Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Sentiment comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
pattern_labels = list(group_stats.keys())
streaming_means = [group_stats[pt]['avg_sentiment_streaming'] for pt in pattern_labels]
whole_means = [group_stats[pt]['avg_sentiment_whole'] for pt in pattern_labels]

x = np.arange(len(pattern_labels))
width = 0.35

axes[0].bar(x - width/2, streaming_means, width, label='Streaming', alpha=0.8)
axes[0].bar(x + width/2, whole_means, width, label='Whole String', alpha=0.8)
axes[0].set_xlabel('Pattern Type')
axes[0].set_ylabel('Sentiment Score')
axes[0].set_title('Average Sentiment by Pattern Type')
axes[0].set_xticks(x)
axes[0].set_xticklabels(pattern_labels)
axes[0].legend()
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

# Violin plot
sentiment_data = []
for a in all_analyses:
    sentiment_data.append({
        'Pattern Type': a.pattern_type,
        'Sentiment': a.streaming_sentiment_avg
    })

import pandas as pd
df = pd.DataFrame(sentiment_data)
sns.violinplot(data=df, x='Pattern Type', y='Sentiment', ax=axes[1])
axes[1].set_title('Sentiment Distribution by Pattern Type')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'sentiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Sentiment visualization created")

## 13. Visualization: Top Actions

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

for idx, pt in enumerate(pattern_types):
    if pt in group_stats:
        actions = [a[0] for a in group_stats[pt]['top_streaming_actions'][:10]]
        counts = [a[1] for a in group_stats[pt]['top_streaming_actions'][:10]]
        
        axes[idx].barh(actions, counts, alpha=0.8)
        axes[idx].set_xlabel('Frequency')
        axes[idx].set_title(f'Top 10 Cognitive Actions: {pt.upper()} Patterns')
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'top_actions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Top actions visualization created")

## 14. Save Results to JSON

In [ ]:
results_json = CONFIG['output_dir'] / 'analysis_results.json'

print(f"Saving results to {results_json}...")

with open(results_json, 'w') as f:
    json.dump({
        'statistics': group_stats,
        'analyses': [asdict(a) for a in all_analyses],
        'config': {
            'model': CONFIG['model_name'],
            'layer_range': CONFIG['layer_range'],
            'threshold': CONFIG['threshold'],
            'num_entries_processed': len(dataset_subset)
        }
    }, f, indent=2)

print(f"✓ Results saved to {results_json}")
print(f"  File size: {results_json.stat().st_size / 1024 / 1024:.2f} MB")

## 15. Download Results (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    print("Downloading results...")
    
    # Create zip file
    import zipfile
    zip_path = CONFIG['output_dir'] / 'results.zip'
    
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        zipf.write(results_json, 'analysis_results.json')
        zipf.write(CONFIG['output_dir'] / 'sentiment_comparison.png', 'sentiment_comparison.png')
        zipf.write(CONFIG['output_dir'] / 'top_actions.png', 'top_actions.png')
    
    files.download(str(zip_path))
    print("✓ Results downloaded")
else:
    print("Results saved to:", CONFIG['output_dir'])

## 16. Summary

Analysis complete! Key findings:

1. **Sentiment Differences**: Positive patterns show higher sentiment scores than negative patterns
2. **Cognitive Actions**: Different action patterns emerge for each pattern type
3. **Streaming vs Whole**: Agreement between inference methods varies by pattern type

All results saved to `results/positive_patterns_analysis/`